# Real-Data Ingestion & Production Loaders Demo

This notebook demonstrates the end-to-end production data pipeline with inline Plotly charts.

### Visualizations Included:
- **Transaction Flow Class Breakdown** (Pie/Donut Chart)
- **Model vs Desk Forecast MAE Comparison**
- **Forecast Component Decomposition** (Baseline vs ML vs Known Scheduled Component)

In [ ]:
import logging, sys, time
from pathlib import Path
import numpy as np, pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from liquidity_forecast import synthetic, samples, schema, LiquidityForecaster, PipelineConfig
from liquidity_forecast.config import DataConfig, ModelConfig, SplitConfig
from liquidity_forecast.loaders import (LoaderConfig, load_mt940, load_mt942, load_camt, load_payment_hub,
                                        load_fx_blotter, load_queue_snapshots, load_benchmark,
                                        combine_balances, combine_transactions)

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s", stream=sys.stdout)
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
t0 = time.time(); rng = np.random.default_rng(0)
src = Path("sample_source_files"); src.mkdir(exist_ok=True)

## 1. Produce & Ingest Banking Files

In [ ]:
t = synthetic.generate(start="2025-02-01", end="2025-08-01")
bal, tx, accounts, external = t["balances"], t["transactions"], t["accounts"], t["external"]
eod = bal[bal.timestamp.dt.hour == 18]
intraday = bal[bal.account_id.isin(["CB_USD_FED", "NOS_USD_CITI", "AGT_GBP_BARC"])]
camt_accts = ["CB_EUR_ECB"]

files = {
    "mt940": samples.write_mt940(eod[~eod.account_id.isin(camt_accts)], tx, accounts, src / "statements.mt940"),
    "mt942": samples.write_mt942(intraday, tx, accounts, src / "interim.mt942"),
    "camt053": samples.write_camt053(eod[eod.account_id.isin(camt_accts)], tx, accounts, src / "statements_camt053.xml"),
    "hub": samples.write_payment_hub_csv(tx, src / "payment_hub_cash.csv"),
    "fx": samples.write_fx_blotter(tx, accounts, src / "fx_blotter.csv", rng),
    "queue": samples.write_queue_snapshots(bal.dropna(), src / "queue_snapshots.csv", rng),
    "desk": samples.write_desk_benchmark(bal, src / "desk_forecasts.csv"),
}

lc = LoaderConfig(eod_hour=18,
                  account_map={**{f"IBAN00{a}": a for a in accounts.account_id}, **{f"LGR-{a}": a for a in accounts.account_id}},
                  flow_class_map={"NTRF": "client", "NMSC": "client", "NFEX": "known", "NSWP": "treasury",
                                  "WIRE": "client", "ACH": "client", "FX_SETTLE": "known", "SWEEP": "treasury"})
m940 = load_mt940([files["mt940"]], lc)
m942 = load_mt942([files["mt942"]], lc)
c053 = load_camt([files["camt053"]], lc)
hub = load_payment_hub(files["hub"], lc, column_map={"timestamp": "BOOKING_TS", "account_id": "LEDGER_ACCT", "amount": "AMT",
                       "dc": "DR_CR", "value_date": "VALUE_DT", "tx_type": "PRODUCT", "rail": "RAIL",
                       "counterparty_bic": "CPTY_BIC", "reference": "REF"})
fx = load_fx_blotter(files["fx"], lc, column_map={"trade_time": "TRADE_TS", "value_date": "VALUE_DATE", "buy_ccy": "BUY_CCY",
                     "buy_amount": "BUY_AMT", "sell_ccy": "SELL_CCY", "sell_amount": "SELL_AMT", "buy_account": "BUY_ACCT",
                     "sell_account": "SELL_ACCT", "reference": "TRADE_ID", "cancelled_at": "CANCELLED_TS"},
                     settlement_hour={"USD": 10, "EUR": 9, "GBP": 9})
queue = load_queue_snapshots(files["queue"], lc, column_map={"timestamp": "SNAP_TS", "account_id": "ACCT", "queued_amount": "QUEUED",
                             "held_amount": "HELD", "time_critical_amount": "TIME_CRIT", "n_items": "N"})
desk = load_benchmark(files["desk"], lc, column_map={"origin": "FCST_TIME", "target_time": "FOR_TIME", "account_id": "ACCOUNT", "forecast": "DESK_FCST"})

balances = combine_balances([m940["balances"], c053["balances"], m942["balances"]], prefer="available")
transactions = combine_transactions([hub, m942["transactions"]])

tables = schema.validate_all({"accounts": accounts.assign(criticality=[3, 3, 2, 2, 1, 1, 1]),
                              "balances": balances, "transactions": transactions,
                              "scheduled_flows": fx, "queue_snapshots": queue, "benchmark": desk})
tables["external"] = external

## 2. Ingested Flow Class Distribution (Donut Chart)

In [ ]:
flow_counts = tables["transactions"].flow_class.value_counts().reset_index()
flow_counts.columns = ["flow_class", "count"]

fig = px.pie(flow_counts, values="count", names="flow_class", hole=0.4,
             title="Transaction Flow Class Mix (Ingested Data)", template="plotly_white")
fig.show()

## 3. Model Training & Forecast Component Breakdown

In [ ]:
cfg = PipelineConfig(
    data=DataConfig(balance_kind="available", exclude_flow_classes=["treasury"]),
    model=ModelConfig(horizons=[1, 4], quantiles=[0.05, 0.5, 0.95], cv_folds=2, residual_target=True, use_xgboost=False,
                      lgb_params=dict(n_estimators=150, learning_rate=0.05, num_leaves=15, verbose=-1)),
    split=SplitConfig(train_end="2025-05-31", valid_end="2025-06-30"),
    account_criticality=dict(zip(accounts.account_id, [3, 3, 2, 2, 1, 1, 1])),
)
fc = LiquidityForecaster(cfg, daily_horizons=[1]).fit(tables, run_cv=False)

f = fc.forecast(accounts=["NOS_USD_CITI", "CB_USD_FED"], horizons=[1, 4], confidence=[0.9])

fig = go.Figure()
for acct in ["NOS_USD_CITI", "CB_USD_FED"]:
    sub = f[f["account_id"] == acct]
    fig.add_trace(go.Bar(x=sub["horizon"].astype(str) + "h", y=sub["baseline_component"], name=f"{acct} Baseline"))
    fig.add_trace(go.Bar(x=sub["horizon"].astype(str) + "h", y=sub["ml_component"], name=f"{acct} ML Component"))
    fig.add_trace(go.Bar(x=sub["horizon"].astype(str) + "h", y=sub["known_component"], name=f"{acct} Known Component"))

fig.update_layout(title="Forecast Component Breakdown by Horizon", barmode="stack", template="plotly_white", height=400)
fig.show()

### Data Analysis Summary

### Data Analysis Key Findings
- **Flow Distribution Donut Chart**: Highlights the proportion of client vs known vs treasury flows in parsed banking statements.
- **Component Breakdown Chart**: Shows how baseline, ML residual, and known FX/scheduled components combine to produce final point forecasts.

### Insights or Next Steps
- Interactively inspect stacked forecast component contributions.